## Lab 2 — OOP Refactor & FastAPI CRUD API


## 0 — Setup

In [6]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', 'fastapi', 'uvicorn[standard]', 'httpx', 'pydantic', '-q'])

import json, pathlib

servers_data = [
    {"id": 1, "name": "api-prod-1", "host": "10.0.0.1", "port": 8080, "status": "unknown", "tags": []},
    {"id": 2, "name": "api-prod-2", "host": "10.0.0.2", "port": 8080, "status": "unknown", "tags": []},
    {"id": 3, "name": "db-prod", "host": "10.0.0.3", "port": 5432, "status": "unknown", "tags": []}
]

pathlib.Path('servers.json').write_text(json.dumps(servers_data, indent=2), encoding='utf-8')
print('✅ Dependencies installed and servers.json created')

✅ Dependencies installed and servers.json created


---
## Task 1 — Model the Server with a Dataclass

### 📖 Concept: `@dataclass`

A `dataclass` auto-generates `__init__`, `__repr__`, and `__eq__` from field annotations. Use it for data containers that don't need complex logic.

```python
from dataclasses import dataclass, field

@dataclass
class Config:
    host: str
    port: int = 8080
    tags: list[str] = field(default_factory=list)  # mutable defaults need field()

cfg = Config('localhost')
print(cfg)  # Config(host='localhost', port=8080, tags=[])
```

> Never use `tags: list = []` as a default — that creates one shared list for all instances. Always use `field(default_factory=list)`.

In [7]:
from dataclasses import dataclass, field

@dataclass
class Server:
    id: int
    name: str
    host: str
    port: int
    status: str = 'unknown'
    tags: list[str] = field(default_factory=list)

    def base_url(self) -> str:
        return f'http://{self.host}:{self.port}'


# Test it
s = Server(id=1, name='api-prod-1', host='10.0.0.1', port=8080)
print(s)
print(s.base_url())

Server(id=1, name='api-prod-1', host='10.0.0.1', port=8080, status='unknown', tags=[])
http://10.0.0.1:8080


---
## Task 2 — ConfigLoader Class

### 📖 Concept: Classes with logging

A class groups related data (the file path) and behaviour (the `load()` method). Use `logging` instead of `print()` for anything that goes to production.

```python
import logging
logger = logging.getLogger(__name__)
logger.info('Loading %d items', count)   # use %s placeholders, not f-strings
```

In [10]:
import json
import logging
import pathlib

logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')
logger = logging.getLogger(__name__)


class ConfigError(ValueError):
    """Raised when config loading fails."""
    pass


class ConfigLoader:
    """Loads server configuration from a JSON file."""

    def __init__(self, path: str):
        self.path = pathlib.Path(path)

    def load(self) -> list[Server]:
        """Parse the config file and return Server objects."""
        try:
            raw = self.path.read_text(encoding='utf-8')
            data = json.loads(raw)
            servers = [Server(**item) for item in data]
            logger.info('Loaded %d server(s) from %s', len(servers), self.path)
            return servers
        except FileNotFoundError as exc:
            raise ConfigError(f'Config file not found: {self.path}') from exc
        except json.JSONDecodeError as exc:
            raise ConfigError(f'Invalid JSON in {self.path}: {exc}') from exc


# Test it
loader = ConfigLoader('servers.json')
server_objects = loader.load()
print(server_objects)

2026-06-29 14:41:04,376 [INFO] Loaded 3 server(s) from servers.json


[Server(id=1, name='api-prod-1', host='10.0.0.1', port=8080, status='unknown', tags=[]), Server(id=2, name='api-prod-2', host='10.0.0.2', port=8080, status='unknown', tags=[]), Server(id=3, name='db-prod', host='10.0.0.3', port=5432, status='unknown', tags=[])]


---
## Task 3 — HealthChecker Class (async)

### 📖 Concept: `async`/`await` and `asyncio.gather()`

An `async def` function is a coroutine — it can pause and yield control while waiting for I/O (network, disk). This lets Python handle other work in the meantime.

`asyncio.gather()` runs multiple coroutines **concurrently** — instead of checking 10 servers one by one (10 × 1s = 10s), it checks all 10 at once (~1s total).

```python
import asyncio, httpx

async def fetch(url: str) -> int:
    async with httpx.AsyncClient() as client:
        resp = await client.get(url)
        return resp.status_code

# Run one coroutine
code = asyncio.run(fetch('https://example.com'))

# Run many concurrently
codes = asyncio.run(asyncio.gather(fetch('https://a.com'), fetch('https://b.com')))
```

In [9]:
import asyncio
import time
import httpx


class HealthChecker:
    """Checks server health over HTTP asynchronously."""

    def __init__(self, timeout: float = 5.0, degraded_threshold_ms: float = 500.0):
        self.timeout = timeout
        self.degraded_threshold_ms = degraded_threshold_ms

    async def check(self, server: Server) -> Server:
        start = time.time()
        try:
            async with httpx.AsyncClient(timeout=self.timeout) as client:
                resp = await client.get(f'{server.base_url()}/health')
            elapsed_ms = (time.time() - start) * 1000
            if resp.status_code == 200 and elapsed_ms <= self.degraded_threshold_ms:
                server.status = 'UP'
            elif resp.status_code == 200:
                server.status = 'DEGRADED'
            else:
                server.status = 'DEGRADED'
        except Exception:
            server.status = 'DOWN'
        return server

    async def check_all(self, servers: list[Server]) -> list[Server]:
        tasks = [self.check(server) for server in servers]
        return await asyncio.gather(*tasks)


# Quick test with a real server
checker = HealthChecker()
test_server = Server(id=99, name='httpbin', host='httpbin.org', port=443)
# We can't use asyncio.run() inside Jupyter — use await directly:
result = await checker.check(test_server)
print(result)

2026-06-29 14:40:59,315 [INFO] HTTP Request: GET http://httpbin.org:443/health "HTTP/1.1 400 Bad Request"


Server(id=99, name='httpbin', host='httpbin.org', port=443, status='DEGRADED', tags=[])


---
## Task 4 — FastAPI CRUD API

### 📖 Concept: Pydantic models for validation

FastAPI uses Pydantic models to validate incoming request bodies and serialise responses automatically.

```python
from pydantic import BaseModel, Field

class ServerIn(BaseModel):
    name: str
    port: int = Field(default=8080, ge=1, le=65535)  # validated automatically
```

FastAPI raises HTTP 422 automatically if validation fails — no manual checking needed.

---

The cells below write the full API to disk. Run the terminal command after to test it.

open a terminal and run:

  "uvicorn lab2_main:app --reload --port 8000"

  
Then visit: http://localhost:8000/docs